# 03 — Sklearn Models

Train and evaluate 5 sklearn regression models on the preprocessed PV data:
1. Linear Regression (baseline)
2. Ridge Regression
3. Random Forest
4. XGBoost
5. SVR

In [2]:
import numpy as np
import pandas as pd
import time
import warnings
from pathlib import Path
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    print("⚠ xgboost not installed, will use GradientBoosting from sklearn instead")
    from sklearn.ensemble import GradientBoostingRegressor
    HAS_XGB = False

In [3]:
# ── Paths ──
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "data" / "processed"
RESULTS_DIR = BASE_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load Data

In [4]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

TARGET = "PV_normalized"
feature_cols = [c for c in train_df.columns if c != TARGET]

X_train = train_df[feature_cols].values
y_train = train_df[TARGET].values
X_test  = test_df[feature_cols].values
y_test  = test_df[TARGET].values

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"Features: {len(feature_cols)}")

X_train: (3047834, 55)
X_test:  (873849, 55)
Features: 55


In [6]:
# Scale features for models that need it (SVR, Ridge)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
print("Features scaled")

Features scaled


## 2. Define Models & Evaluation

In [7]:
def evaluate(y_true, y_pred, model_name):
    """Compute regression metrics."""
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    # MAPE: avoid div by zero on nighttime (PV=0) rows
    mask = y_true > 0.01
    if mask.sum() > 0:
        mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    else:
        mape = np.nan
    print(f"  {model_name:25s} | MAE={mae:.5f} | RMSE={rmse:.5f} | R²={r2:.4f} | MAPE={mape:.2f}%")
    return {"Model": model_name, "MAE": mae, "RMSE": rmse, "R2": r2, "MAPE": mape}

In [8]:
# Subsample for SVR (3M rows is too large)
# SVR is O(n²) — use ~50k rows for training
SVR_SAMPLE = 50_000
np.random.seed(42)
svr_idx = np.random.choice(len(X_train_scaled), size=min(SVR_SAMPLE, len(X_train_scaled)), replace=False)
X_train_svr = X_train_scaled[svr_idx]
y_train_svr = y_train[svr_idx]
print(f"SVR subsample: {X_train_svr.shape[0]:,} rows")

SVR subsample: 50,000 rows


## 3. Train & Evaluate Models

In [9]:
results = []

models = [
    ("Linear Regression",  LinearRegression(),                              X_train_scaled, X_test_scaled, y_train),
    ("Ridge Regression",   Ridge(alpha=1.0),                                X_train_scaled, X_test_scaled, y_train),
    ("Random Forest",      RandomForestRegressor(n_estimators=100, max_depth=15, n_jobs=-1, random_state=42), X_train, X_test, y_train),
]

if HAS_XGB:
    models.append(("XGBoost", XGBRegressor(n_estimators=200, max_depth=8, learning_rate=0.1,
                                            n_jobs=-1, random_state=42, verbosity=0),
                   X_train, X_test, y_train))
else:
    models.append(("Gradient Boosting", GradientBoostingRegressor(n_estimators=200, max_depth=8,
                                                                   learning_rate=0.1, random_state=42),
                   X_train, X_test, y_train))

models.append(("SVR (RBF)", SVR(kernel="rbf", C=1.0, epsilon=0.01), X_train_svr, X_test_scaled, y_train_svr))

In [11]:
print("Training and evaluating models...\n")

for name, model, X_tr, X_te, y_tr in models:
    print(f"  -> Training {name}...")
    t0 = time.time()
    model.fit(X_tr, y_tr)
    train_time = time.time() - t0
    
    y_pred = model.predict(X_te)
    y_pred = np.clip(y_pred, 0, 1)  # Clip predictions to valid range
    
    res = evaluate(y_test, y_pred, name)
    res["Train Time (s)"] = round(train_time, 1)
    results.append(res)
    print(f"    (trained in {train_time:.1f}s)\n")

Training and evaluating models...

  -> Training Linear Regression...
  Linear Regression         | MAE=0.02972 | RMSE=0.05078 | R²=0.9465 | MAPE=42.24%
    (trained in 16.3s)

  -> Training Ridge Regression...
  Ridge Regression          | MAE=0.02972 | RMSE=0.05078 | R²=0.9465 | MAPE=42.24%
    (trained in 2.8s)

  -> Training Random Forest...
  Random Forest             | MAE=0.01302 | RMSE=0.03434 | R²=0.9755 | MAPE=19.71%
    (trained in 1148.8s)

  -> Training XGBoost...
  XGBoost                   | MAE=0.01279 | RMSE=0.03268 | R²=0.9778 | MAPE=18.63%
    (trained in 49.2s)

  -> Training SVR (RBF)...
  SVR (RBF)                 | MAE=0.01699 | RMSE=0.03823 | R²=0.9697 | MAPE=22.54%
    (trained in 1007.8s)



## 4. Results Summary

In [12]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values("R2", ascending=False).reset_index(drop=True)
results_df

,Model,MAE,RMSE,R2,MAPE,Train Time (s)
0,XGBoost,0.012787,0.032676,0.977836,18.633633,49.2
1,Random Forest,0.013019,0.034340,0.975521,19.707382,1148.8
2,SVR (RBF),0.016990,0.038234,0.969654,22.537968,1007.8
3,Linear Regression,0.029718,0.050776,0.946479,42.244303,17.6
4,Linear Regression,0.029718,0.050776,0.946479,42.244303,16.3
5,Ridge Regression,0.029718,0.050777,0.946479,42.244312,2.8


In [13]:
# Save results
results_df.to_csv(RESULTS_DIR / "sklearn_results.csv", index=False)
print(f"Saved to {RESULTS_DIR / 'sklearn_results.csv'}")

Saved to /home/gujjavarshith/Documents/SEM 6/FSD 2/Solar_energy_prediction/results/sklearn_results.csv


In [15]:
best = results_df.iloc[0]
print(f"\n Best sklearn model: {best['Model']}")
print(f"   R² = {best['R2']:.4f}")
print(f"   MAE = {best['MAE']:.5f}")
print(f"   RMSE = {best['RMSE']:.5f}")


 Best sklearn model: XGBoost
   R² = 0.9778
   MAE = 0.01279
   RMSE = 0.03268


In [19]:
import joblib

best_model_name = best['Model']
best_model_obj = next(model for name, model, *_ in models if name == best_model_name)


model_filename = f"best_{best_model_name.replace(' ', '_').lower()}.joblib"
model_path = BASE_DIR/ "models"/"sklearn" / model_filename
joblib.dump(best_model_obj, model_path)
print(f"Saved best model ({best_model_name}) to: {model_path}")

scaler_path = BASE_DIR/ "models"/"sklearn" / "scaler.joblib"
joblib.dump(scaler, scaler_path)
print(f" Saved StandardScaler to: {scaler_path}")


Saved best model (XGBoost) to: /home/gujjavarshith/Documents/SEM 6/FSD 2/Solar_energy_prediction/models/sklearn/best_xgboost.joblib
 Saved StandardScaler to: /home/gujjavarshith/Documents/SEM 6/FSD 2/Solar_energy_prediction/models/sklearn/scaler.joblib
